## Dream 2000 Smart Watches Scrapping

In [ ]:
import selenium 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
import time

## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")


## -- Smart_Watches List --
Smart_Watches = []


## -- Scraping --
driver = webdriver.Chrome(options=options)
driver.get("https://dream2000.com/en/collections/wearables")
time.sleep(5)

page = 1

while True:

    page_products = 0

    ## -- Product Cards --
    cards = driver.find_elements(By.CSS_SELECTOR,'.product-card__wrapper.color-scheme-theme.anm-reveal-item.overflow-hidden')

    for card in cards:

        try:
            Product = card.find_element(By.CSS_SELECTOR, 'a[href*="/en/products/"]')

            Name = Product.get_attribute('aria-label')
            Link = Product.get_attribute('href')

            Image = card.find_element(By.CSS_SELECTOR, 'img.product-card__image').get_attribute('src')

        except NoSuchElementException:
            continue

        ## -- Price --
        try:
            Price = card.find_element(By.CSS_SELECTOR, '.price').text

        except NoSuchElementException:
            Price = "Product Not Available"

        ## -- Stock --
        try:
            Stock = card.find_element(By.CSS_SELECTOR,'[data-inventory-status]').get_attribute('data-inventory-status')

        except NoSuchElementException:
            Stock = "Product Not Available"

        Smart_Watches.append({
            'Name': Name,
            'Link': Link,
            'Image': Image,
            'Price': Price,
            'Stock': Stock
        })

        page_products += 1

    ## -- Page Summary --

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Smart_Watches)}"
    )

    ## -- Next Page --
    next_buttons = driver.find_elements(
        By.CSS_SELECTOR, "a[aria-label='Next']"
    )

    if not next_buttons:
        print("No Next page. Scraping finished.")
        break

    next_url = next_buttons[0].get_attribute('href')

    if not next_url or next_url == driver.current_url:
        print("Invalid or unchanged Next URL.")
        break

    print("Navigating to:", next_url)

    driver.get(next_url)

    page += 1
    time.sleep(5)


## -- Final Result --
print(f"\nScraping completed. Total products: {len(Smart_Watches)}")


In [ ]:
Smart_Watches

## Import To CSV & Parquet File

In [2]:
import pandas as pd

In [4]:
df_Smart_Watches = pd.DataFrame(Smart_Watches) 

In [5]:
df_Smart_Watches.head()

,Name,Link,Image,Price,Stock
0,Xiaomi Redmi Watch 5 Lite,https://dream2000.com/en/products/redmi-watch-...,https://dream2000.com/cdn/shop/files/2025-01-1...,"LE 3,140.00\nLE 3,299.00",in-stock
1,"Oraimo Watch 6 Lite Smart Watch , 2.01 Inch Bl...",https://dream2000.com/en/products/oraimo-watch...,https://dream2000.com/cdn/shop/files/1_0d0a5e4...,LE 900.00\nLE 949.00,in-stock
2,Huawei Watch Fit 4 SYA-B09,https://dream2000.com/en/products/huawei-watch...,https://dream2000.com/cdn/shop/files/2025-05-2...,"LE 7,280.00\nLE 7,699.00",in-stock
3,"Itel Fit 020 Smart Watch, Bluetooth 5.4, 2.04 ...",https://dream2000.com/en/products/itel-fit-020...,https://dream2000.com/cdn/shop/files/dfddfdfdf...,LE 960.00\nLE 999.00,in-stock
4,Mibro Smart Watch Lite 3,https://dream2000.com/en/products/mibro-smart-...,https://dream2000.com/cdn/shop/files/2025-02-1...,"LE 2,499.00\nLE 2,700.00",in-stock


In [6]:
df_Smart_Watches.to_csv("Smart_Watches.csv" , index = False)
df_Smart_Watches.to_parquet("Smart_Watches.parquet" , index = False)